# Análisis energético: modelos básico, parcial y avanzado

Este notebook demuestra una arquitectura con **tres pipelines independientes** para evitar inventar datos opcionales:

- El **modelo básico** usa exclusivamente las cinco variables exigidas por el proyecto.
- El **modelo parcial** aprovecha de uno a cuatro datos avanzados y conserva los demás como ausentes.
- El **modelo avanzado** se usa solamente cuando todos los datos avanzados fueron proporcionados.
- No se imputan personas, área, aire acondicionado, consumo anterior ni días facturados.
- Ningún nivel reemplaza datos faltantes con medianas o ceros supuestos.

> La aplicación ya utiliza esta arquitectura mediante tres artefactos separados. Este notebook conserva una exportación experimental protegida y no sobrescribe los modelos productivos. El dataset es simulado y no representa una certificación energética oficial.


## Arquitectura propuesta

```text
Solicitud
   │
   ├─ 0 datos avanzados ─→ Modelo básico
   ├─ 1 a 4 datos avanzados ─→ Modelo parcial
   └─ 5 datos avanzados ─→ Modelo avanzado
```

No se convierte un dato desconocido en cero ni se reemplaza por una mediana. El modelo parcial usa manejo nativo de ausencias: aprovecha los valores presentes y crea ramas específicas para los faltantes.


In [1]:
%pip install -q pandas==2.2.2 scikit-learn==1.6.1 joblib==1.5.3 matplotlib seaborn

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

CURRENT_DIR = Path.cwd()
if EN_COLAB:
    MY_DRIVE = Path('/content/drive/MyDrive')
    CANDIDATOS = [
        MY_DRIVE / 'Notebook Hackathon' / 'analisis-de-energia',
        MY_DRIVE / 'ciencia-datos' / 'analisis-de-energia',
        MY_DRIVE / 'analisis-de-energia',
    ]
elif CURRENT_DIR.name == 'analisis-de-energia':
    CANDIDATOS = [CURRENT_DIR]
else:
    CANDIDATOS = [
        CURRENT_DIR / 'ciencia-datos' / 'analisis-de-energia',
        CURRENT_DIR / 'analisis-de-energia',
    ]

ANALYSIS_DIR = next((ruta for ruta in CANDIDATOS if ruta.is_dir()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('No se encontró la carpeta analisis-de-energia.')

DATASET_PATH = ANALYSIS_DIR / 'data' / 'raw' / 'consumo_energetico.csv'
PRODUCTION_MODEL_PATH = ANALYSIS_DIR / 'models' / 'modelo_energia.joblib'
EXPERIMENT_DIR = ANALYSIS_DIR / 'models' / 'experimentos_tres_niveles'
BASIC_MODEL_PATH = EXPERIMENT_DIR / 'modelo_energia_basico.joblib'
PARTIAL_MODEL_PATH = EXPERIMENT_DIR / 'modelo_energia_parcial.joblib'
ADVANCED_MODEL_PATH = EXPERIMENT_DIR / 'modelo_energia_avanzado.joblib'
METADATA_PATH = EXPERIMENT_DIR / 'metadata_tres_niveles.json'

print('Entorno:', 'Google Colab' if EN_COLAB else 'Local')
print('Dataset:', DATASET_PATH)
print('Modelo productivo protegido:', PRODUCTION_MODEL_PATH)
print('Salidas experimentales:', EXPERIMENT_DIR)

Mounted at /content/drive
Entorno: Google Colab
Dataset: /content/drive/MyDrive/Notebook Hackathon/analisis-de-energia/data/raw/consumo_energetico.csv
Modelo productivo protegido: /content/drive/MyDrive/Notebook Hackathon/analisis-de-energia/models/modelo_energia.joblib
Salidas experimentales: /content/drive/MyDrive/Notebook Hackathon/analisis-de-energia/models/experimentos_tres_niveles


## 1. Carga y validación del dataset

La misma partición de entrenamiento y prueba se utiliza para comparar los tres niveles de forma justa.


In [3]:
df = pd.read_csv(DATASET_PATH)

MAPEO_CATEGORIAS = {'Ineficiente': 0, 'Moderado': 1, 'Eficiente': 2}
MAPEO_CATEGORIAS_INVERSO = {valor: clave for clave, valor in MAPEO_CATEGORIAS.items()}

if not set(df['categoria'].unique()).issubset(MAPEO_CATEGORIAS):
    raise ValueError('El dataset contiene categorías desconocidas.')

print('Dimensiones:', df.shape)
display(df.head())
display(df['categoria'].value_counts())

Dimensiones: (8000, 23)


,consumo_kwh,uso_horario_pico,cantidad_equipos,tipo_inmueble,horas_alto_consumo,cantidad_personas,area_m2,equipos_alto_consumo,equipos_medio_consumo,equipos_bajo_consumo,...,consumo_diario,consumo_por_persona,consumo_por_m2,variacion_mensual,proporcion_equipos_alto_consumo,proporcion_equipos_medio_consumo,proporcion_equipos_bajo_consumo,carga_relativa_equipos,puntaje_ineficiencia,categoria
0,977.23,True,27,Comercio,8,15,54.0,1,9,17,...,33.6976,65.1487,18.0969,0.1075,0.0370,0.3333,0.6296,0.2167,0.2862,Moderado
1,528.54,False,13,Apartamento,12,5,143.0,2,6,5,...,17.0497,105.7080,3.6961,0.3011,0.1538,0.4615,0.3846,0.3538,0.2094,Eficiente
2,888.54,True,21,Comercio,11,1,279.0,1,5,15,...,28.6626,888.5400,3.1847,0.0565,0.0476,0.2381,0.7143,0.2024,0.2887,Moderado
3,699.05,False,10,Apartamento,19,3,180.0,1,3,6,...,24.9661,233.0167,3.8836,-0.0607,0.1000,0.3000,0.6000,0.2650,0.2578,Moderado
4,609.28,False,12,Casa,21,6,324.0,0,3,9,...,21.0097,101.5467,1.8805,0.2005,0.0000,0.2500,0.7500,0.1625,0.2579,Moderado


,count
categoria,
Moderado,2960
Eficiente,2640
Ineficiente,2400


## 2. Variables separadas por nivel

El modelo básico coincide literalmente con la entrada solicitada por el proyecto:

```json
{
  "consumo_kwh": 420,
  "uso_horario_pico": true,
  "cantidad_equipos": 10,
  "tipo_inmueble": "Casa",
  "horas_alto_consumo": 8
}
```

La distribución de equipos puede seguir utilizándose en el modelo de recomendaciones. No se incluye aquí para que el clasificador básico cumpla exactamente con las cinco variables requeridas.


In [4]:
BASIC_FEATURES = [
    'consumo_kwh',
    'uso_horario_pico',
    'cantidad_equipos',
    'tipo_inmueble',
    'horas_alto_consumo',
]

ADVANCED_USER_FIELDS = [
    'cantidad_personas',
    'area_m2',
    'horas_aire_acondicionado',
    'consumo_mes_anterior_kwh',
    'dias_facturados',
]

DERIVED_ADVANCED_FEATURES = [
    'consumo_por_persona',
    'consumo_por_m2',
    'variacion_mensual',
]

ADVANCED_FEATURES = (
    BASIC_FEATURES + ADVANCED_USER_FIELDS + DERIVED_ADVANCED_FEATURES
)

DEPENDENCIAS_AVANZADAS = {
    'cantidad_personas': ['consumo_por_persona'],
    'area_m2': ['consumo_por_m2'],
    'horas_aire_acondicionado': [],
    'consumo_mes_anterior_kwh': ['variacion_mensual'],
    'dias_facturados': [],
}

columnas_requeridas = set(ADVANCED_FEATURES + ['categoria'])
columnas_faltantes = sorted(columnas_requeridas - set(df.columns))
if columnas_faltantes:
    raise ValueError(f'Faltan columnas en el dataset: {columnas_faltantes}')

if df[ADVANCED_FEATURES].isna().any().any():
    raise ValueError('El entrenamiento avanzado requiere datos completos; no se imputarán valores.')

print('Variables básicas:', BASIC_FEATURES)
print('Variables avanzadas:', ADVANCED_FEATURES)

Variables básicas: ['consumo_kwh', 'uso_horario_pico', 'cantidad_equipos', 'tipo_inmueble', 'horas_alto_consumo']
Variables avanzadas: ['consumo_kwh', 'uso_horario_pico', 'cantidad_equipos', 'tipo_inmueble', 'horas_alto_consumo', 'cantidad_personas', 'area_m2', 'horas_aire_acondicionado', 'consumo_mes_anterior_kwh', 'dias_facturados', 'consumo_por_persona', 'consumo_por_m2', 'variacion_mensual']


## 3. Partición reproducible


In [5]:
y = df['categoria'].map(MAPEO_CATEGORIAS).astype(int)
indices_train, indices_test = train_test_split(
    df.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train = y.loc[indices_train]
y_test = y.loc[indices_test]

print('Registros de entrenamiento:', len(indices_train))
print('Registros de prueba:', len(indices_test))

Registros de entrenamiento: 6400
Registros de prueba: 1600


## 4. Entrenamiento independiente

Cada nivel tiene su propio preprocesamiento y su propio clasificador. Los pipelines no contienen `SimpleImputer`: si una columna del modelo seleccionado está vacía, el error debe detectarse antes de predecir.


In [6]:
MODELOS_CANDIDATOS = {
    'regresion_logistica': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'arbol_decision': DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=3,
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=14, min_samples_leaf=3,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

def crear_pipeline(columnas, clasificador):
    categoricas = ['tipo_inmueble']
    numericas = [columna for columna in columnas if columna not in categoricas]
    preprocesador = ColumnTransformer([
        ('numericas', StandardScaler(), numericas),
        ('categoricas', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoricas),
    ], sparse_threshold=0)
    return Pipeline([
        ('preprocessor', preprocesador),
        ('classifier', clone(clasificador)),
    ])

def entrenar_nivel(columnas):
    X_train = df.loc[indices_train, columnas]
    X_test = df.loc[indices_test, columnas]
    resultados = {}
    pipelines = {}

    for nombre, clasificador in MODELOS_CANDIDATOS.items():
        pipeline = crear_pipeline(columnas, clasificador)
        pipeline.fit(X_train, y_train)
        predicciones = pipeline.predict(X_test)
        resultados[nombre] = {
            'accuracy': accuracy_score(y_test, predicciones),
            'f1_macro': f1_score(y_test, predicciones, average='macro'),
        }
        pipelines[nombre] = pipeline

    ganador = max(resultados, key=lambda nombre: resultados[nombre]['f1_macro'])
    return ganador, pipelines[ganador], resultados

def crear_datos_parciales(indices, repeticiones, semilla):
    rng = np.random.default_rng(semilla)
    entradas, objetivos = [], []
    campos = list(DEPENDENCIAS_AVANZADAS)
    for _ in range(repeticiones):
        parcial = df.loc[indices, ADVANCED_FEATURES].copy()
        for indice in parcial.index:
            cantidad_presentes = int(rng.integers(1, len(campos)))
            presentes = set(rng.choice(campos, size=cantidad_presentes, replace=False))
            for campo, derivados in DEPENDENCIAS_AVANZADAS.items():
                if campo not in presentes:
                    parcial.loc[indice, [campo, *derivados]] = np.nan
        entradas.append(parcial.reset_index(drop=True))
        objetivos.append(y.loc[indices].reset_index(drop=True))
    return pd.concat(entradas, ignore_index=True), pd.concat(objetivos, ignore_index=True)

def entrenar_parcial():
    X_train_parcial, y_train_parcial = crear_datos_parciales(indices_train, 2, RANDOM_STATE)
    X_test_parcial, y_test_parcial = crear_datos_parciales(indices_test, 1, RANDOM_STATE + 1)
    modelo = HistGradientBoostingClassifier(
        max_iter=250, max_leaf_nodes=31, learning_rate=0.08,
        l2_regularization=0.2, class_weight='balanced', random_state=RANDOM_STATE
    )
    pipeline = crear_pipeline(ADVANCED_FEATURES, modelo)
    pipeline.fit(X_train_parcial, y_train_parcial)
    predicciones = pipeline.predict(X_test_parcial)
    metricas = {
        'accuracy': accuracy_score(y_test_parcial, predicciones),
        'f1_macro': f1_score(y_test_parcial, predicciones, average='macro'),
    }
    return pipeline, metricas

In [7]:
nombre_basico, modelo_basico, resultados_basicos = entrenar_nivel(BASIC_FEATURES)
modelo_parcial, resultados_parciales = entrenar_parcial()
nombre_avanzado, modelo_avanzado, resultados_avanzados = entrenar_nivel(ADVANCED_FEATURES)

comparacion = pd.DataFrame({
    'modelo_basico': resultados_basicos[nombre_basico],
    'modelo_parcial': resultados_parciales,
    'modelo_avanzado': resultados_avanzados[nombre_avanzado],
}).T
comparacion['algoritmo_ganador'] = [nombre_basico, 'hist_gradient_boosting', nombre_avanzado]
display(comparacion)

,accuracy,f1_macro,algoritmo_ganador
modelo_basico,0.813750,0.818209,regresion_logistica
modelo_parcial,0.789375,0.794826,hist_gradient_boosting
modelo_avanzado,0.821250,0.825283,regresion_logistica


In [8]:
for nivel, modelo, columnas in [
    ('BÁSICO', modelo_basico, BASIC_FEATURES),
    ('AVANZADO', modelo_avanzado, ADVANCED_FEATURES),
]:
    predicciones = modelo.predict(df.loc[indices_test, columnas])
    print(f'\n=== REPORTE {nivel} ===')
    print(classification_report(
        y_test, predicciones,
        labels=[0, 1, 2],
        target_names=['Ineficiente', 'Moderado', 'Eficiente'],
        zero_division=0,
    ))


=== REPORTE BÁSICO ===
              precision    recall  f1-score   support

 Ineficiente       0.87      0.88      0.87       480
    Moderado       0.75      0.75      0.75       592
   Eficiente       0.83      0.83      0.83       528

    accuracy                           0.81      1600
   macro avg       0.82      0.82      0.82      1600
weighted avg       0.81      0.81      0.81      1600


=== REPORTE AVANZADO ===
              precision    recall  f1-score   support

 Ineficiente       0.87      0.86      0.87       480
    Moderado       0.76      0.76      0.76       592
   Eficiente       0.85      0.85      0.85       528

    accuracy                           0.82      1600
   macro avg       0.83      0.82      0.83      1600
weighted avg       0.82      0.82      0.82      1600



## 5. Selección automática sin imputación

Reglas:

1. Los cinco datos básicos siempre son obligatorios.
2. Con cero datos avanzados se usa el modelo básico.
3. Con uno a cuatro datos avanzados se usa el modelo parcial.
4. Con los cinco datos avanzados se usa el modelo avanzado.
5. Ninguna mediana o valor supuesto se inserta en la solicitud.


In [9]:
def preparar_variables_avanzadas(datos):
    preparados = dict(datos)
    preparados['consumo_por_persona'] = (
        preparados['consumo_kwh'] / preparados['cantidad_personas']
    )
    preparados['consumo_por_m2'] = (
        preparados['consumo_kwh'] / preparados['area_m2']
    )
    preparados['variacion_mensual'] = (
        preparados['consumo_kwh'] - preparados['consumo_mes_anterior_kwh']
    ) / preparados['consumo_mes_anterior_kwh']
    return preparados

def preparar_variables_parciales(datos):
    preparados = dict(datos)
    for campo in ADVANCED_USER_FIELDS:
        preparados.setdefault(campo, np.nan)
    preparados['consumo_por_persona'] = (
        preparados['consumo_kwh'] / preparados['cantidad_personas']
        if pd.notna(preparados['cantidad_personas']) else np.nan
    )
    preparados['consumo_por_m2'] = (
        preparados['consumo_kwh'] / preparados['area_m2']
        if pd.notna(preparados['area_m2']) else np.nan
    )
    preparados['variacion_mensual'] = (
        (preparados['consumo_kwh'] - preparados['consumo_mes_anterior_kwh'])
        / preparados['consumo_mes_anterior_kwh']
        if pd.notna(preparados['consumo_mes_anterior_kwh']) else np.nan
    )
    return preparados

def predecir_consumo(datos):
    faltantes_basicos = [
        campo for campo in BASIC_FEATURES
        if campo not in datos or datos[campo] is None
    ]
    if faltantes_basicos:
        raise ValueError(f'Faltan datos básicos obligatorios: {faltantes_basicos}')

    cantidad_avanzados = sum(
        campo in datos and datos[campo] is not None
        for campo in ADVANCED_USER_FIELDS
    )

    if cantidad_avanzados == len(ADVANCED_USER_FIELDS):
        nivel = 'avanzado'
        modelo = modelo_avanzado
        columnas = ADVANCED_FEATURES
        preparados = preparar_variables_avanzadas(datos)
    elif cantidad_avanzados == 0:
        nivel = 'basico'
        modelo = modelo_basico
        columnas = BASIC_FEATURES
        preparados = datos
    else:
        nivel = 'parcial'
        modelo = modelo_parcial
        columnas = ADVANCED_FEATURES
        preparados = preparar_variables_parciales(datos)

    entrada = pd.DataFrame([[preparados[c] for c in columnas]], columns=columnas)
    categoria = int(modelo.predict(entrada)[0])
    probabilidades = modelo.predict_proba(entrada)[0]
    clases = list(modelo.named_steps['classifier'].classes_)

    return {
        'nivel_analisis': nivel,
        'categoria': MAPEO_CATEGORIAS_INVERSO[categoria],
        'probabilidad': round(float(probabilidades[clases.index(categoria)]), 4),
        'variables_utilizadas': columnas,
        'datos_imputados': [],
    }

## 6. Pruebas de comportamiento


In [15]:
entrada_basica = {
    'consumo_kwh': 420,
    'uso_horario_pico': True,
    'cantidad_equipos': 10,
    'tipo_inmueble': 'Casa',
    'horas_alto_consumo': 8,
}

resultado_basico = predecir_consumo(entrada_basica)
assert resultado_basico['nivel_analisis'] == 'basico'
assert resultado_basico['variables_utilizadas'] == BASIC_FEATURES
assert resultado_basico['datos_imputados'] == []
resultado_basico

{'nivel_analisis': 'basico',
 'categoria': 'Moderado',
 'probabilidad': 0.6778,
 'variables_utilizadas': ['consumo_kwh',
  'uso_horario_pico',
  'cantidad_equipos',
  'tipo_inmueble',
  'horas_alto_consumo'],
 'datos_imputados': []}

In [16]:
entrada_avanzada = {
    **entrada_basica,
    'cantidad_personas': 4,
    'area_m2': 120.0,
    'horas_aire_acondicionado': 0.0,
    'consumo_mes_anterior_kwh': 380.0,
    'dias_facturados': 30,
}

resultado_avanzado = predecir_consumo(entrada_avanzada)
assert resultado_avanzado['nivel_analisis'] == 'avanzado'
assert resultado_avanzado['datos_imputados'] == []
resultado_avanzado

{'nivel_analisis': 'avanzado',
 'categoria': 'Moderado',
 'probabilidad': 0.579,
 'variables_utilizadas': ['consumo_kwh',
  'uso_horario_pico',
  'cantidad_equipos',
  'tipo_inmueble',
  'horas_alto_consumo',
  'cantidad_personas',
  'area_m2',
  'horas_aire_acondicionado',
  'consumo_mes_anterior_kwh',
  'dias_facturados',
  'consumo_por_persona',
  'consumo_por_m2',
  'variacion_mensual'],
 'datos_imputados': []}

In [17]:
entrada_parcial = {**entrada_basica, 'cantidad_personas': 4}
resultado_parcial = predecir_consumo(entrada_parcial)

assert resultado_parcial['nivel_analisis'] == 'parcial'
assert 'cantidad_personas' in resultado_parcial['variables_utilizadas']
assert resultado_parcial['datos_imputados'] == []
resultado_parcial

{'nivel_analisis': 'parcial',
 'categoria': 'Moderado',
 'probabilidad': 0.6296,
 'variables_utilizadas': ['consumo_kwh',
  'uso_horario_pico',
  'cantidad_equipos',
  'tipo_inmueble',
  'horas_alto_consumo',
  'cantidad_personas',
  'area_m2',
  'horas_aire_acondicionado',
  'consumo_mes_anterior_kwh',
  'dias_facturados',
  'consumo_por_persona',
  'consumo_por_m2',
  'variacion_mensual'],
 'datos_imputados': []}

## 7. Exportación experimental protegida

La exportación está desactivada por defecto. Si se habilita manualmente, los archivos se guardan en `models/experimentos_tres_niveles/`, nunca en la ruta utilizada actualmente por `modelo-api`.


In [19]:
EXPORTAR_MODELOS_EXPERIMENTALES = False

metadata = {
    'version': '2.1.0-experimental',
    'sin_imputacion': True,
    'regla_enrutamiento': (
        'usar básico sin campos avanzados, parcial con uno a cuatro y '        'avanzado con los cinco campos avanzados'
    ),
    'basico': {
        'columnas': BASIC_FEATURES,
        'modelo_seleccionado': nombre_basico,
        'metricas': resultados_basicos[nombre_basico],
    },
    'parcial': {
        'campos_admitidos': ADVANCED_USER_FIELDS,
        'columnas': ADVANCED_FEATURES,
        'modelo_seleccionado': 'hist_gradient_boosting',
        'metricas': resultados_parciales,
        'manejo_ausentes': 'nativo_sin_imputacion',
    },
    'avanzado': {
        'campos_requeridos': ADVANCED_USER_FIELDS,
        'columnas': ADVANCED_FEATURES,
        'modelo_seleccionado': nombre_avanzado,
        'metricas': resultados_avanzados[nombre_avanzado],
    },
}

if EXPORTAR_MODELOS_EXPERIMENTALES:
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(modelo_basico, BASIC_MODEL_PATH)
    joblib.dump(modelo_parcial, PARTIAL_MODEL_PATH)
    joblib.dump(modelo_avanzado, ADVANCED_MODEL_PATH)
    METADATA_PATH.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    print('Modelos experimentales guardados en:', EXPERIMENT_DIR)
else:
    print('Exportación desactivada: no se modificó ningún modelo de la aplicación.')

Exportación desactivada: no se modificó ningún modelo de la aplicación.


## Conclusión

Con esta arquitectura, el análisis básico no supone que el usuario tenga aire acondicionado, cinco personas o un inmueble de cierto tamaño. Esos datos sencillamente no existen para el pipeline básico. El parcial aprovecha de uno a cuatro datos avanzados reales y conserva los demás como ausentes, sin inventarlos. El avanzado entra en operación con los cinco datos avanzados proporcionados por el usuario.

`modelo-api` aplica actualmente la misma regla de enrutamiento y carga artefactos básicos, parciales y avanzados separados. Este notebook permite reproducir el experimento sin sobrescribir por accidente los archivos productivos.


In [14]:
print('=== VERIFICACIÓN FINAL DEL NOTEBOOK ===')
print('Modelo básico usa exactamente cinco variables:', len(BASIC_FEATURES) == 5)
print('Modelo básico sin datos opcionales:', not set(ADVANCED_USER_FIELDS) & set(BASIC_FEATURES))
print('Pipelines sin imputadores:', all('imputer' not in modelo.named_steps for modelo in [modelo_basico, modelo_parcial, modelo_avanzado]))
print('Entrada básica sin imputación:', resultado_basico['datos_imputados'] == [])
print('Entrada parcial usa modelo parcial:', resultado_parcial['nivel_analisis'] == 'parcial')
print('Modelo productivo no es destino de exportación:', PRODUCTION_MODEL_PATH not in [BASIC_MODEL_PATH, PARTIAL_MODEL_PATH, ADVANCED_MODEL_PATH])

=== VERIFICACIÓN FINAL DEL NOTEBOOK ===
Modelo básico usa exactamente cinco variables: True
Modelo básico sin datos opcionales: True
Pipelines sin imputadores: True
Entrada básica sin imputación: True
Entrada parcial usa modelo parcial: True
Modelo productivo no es destino de exportación: True
